# Step 8: Relation-Chain Bridge Expansion Feasibility Check

这个 notebook 用于执行 `Round 1d` 的第一批 relation-chain bridge 扩池。

它的目标不是直接构造新 source set，而是先回答：

- `HotpotQA` 当前 split 里是否存在足够多的 `relation_chain_bridge` 候选题
- 是否值得继续第二批扩池
- 还是应直接判定当前 source benchmark 不足以支持这个 subtype


In [1]:
!pip install -q datasets


In [2]:
import csv
import json
import re
from collections import Counter
from pathlib import Path

from datasets import load_dataset


/Users/mac/miniforge3/envs/mind2web/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 配置路径与批量参数


In [3]:
PROJECT_ROOT_OVERRIDE = ''
CURRENT_SPLIT = 'validation'
RAW_CANDIDATE_SIZE = 15
CURRENT_BRIDGE_SOURCE_SET_ID = 'hp_bridge_set_01'

RELATION_TERMS = [
    'father-in-law',
    'mother-in-law',
    'brother-in-law',
    'sister-in-law',
    'spouse',
    'husband',
    'wife',
    'father',
    'mother',
    'sibling',
    'son',
    'daughter',
]


def detect_project_root():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])

    seen = set()
    checked = []
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        checked.append(str(candidate))
        if (candidate / 'pilot' / 'archive' / 'source_sets_round1.csv').exists() and (candidate / 'results' / '07_round1d_prep' / 'bridge_subtype_pairing_repair.md').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root. Checked: ' + ' | '.join(checked))


PROJECT_ROOT = detect_project_root()
ARCHIVE_DIR = PROJECT_ROOT / 'pilot' / 'archive'
RESULTS_DIR = PROJECT_ROOT / 'results' / '08_relation_chain_bridge_expansion'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_SETS_PATH = ARCHIVE_DIR / 'source_sets_round1.csv'
TAXONOMY_PATH = ARCHIVE_DIR / 'taxonomy_round1.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('CURRENT_SPLIT =', CURRENT_SPLIT)
print('RAW_CANDIDATE_SIZE =', RAW_CANDIDATE_SIZE)


PROJECT_ROOT = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer
RESULTS_DIR = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/08_relation_chain_bridge_expansion
CURRENT_SPLIT = validation
RAW_CANDIDATE_SIZE = 15


## 2. 加载 HotpotQA 与当前 source-side 状态


In [4]:
hotpot = load_dataset('hotpot_qa', 'fullwiki', split=CURRENT_SPLIT)


def read_csv(path):
    with path.open() as f:
        return list(csv.DictReader(f))


source_sets = read_csv(SOURCE_SETS_PATH)
taxonomy_rows = read_csv(TAXONOMY_PATH)
taxonomy_by_task = {row['task_id']: row for row in taxonomy_rows}

bridge_source = next(row for row in source_sets if row['source_set_id'] == CURRENT_BRIDGE_SOURCE_SET_ID)
current_bridge_members = set(bridge_source['member_task_ids'].split('|'))


def task_id_from_index(idx: int) -> str:
    return f'hp_dev_{idx:04d}'


print('hotpot size =', len(hotpot))
print('current bridge source members =', sorted(current_bridge_members))


hotpot size = 7405
current bridge source members = ['hp_dev_1119', 'hp_dev_1668', 'hp_dev_2054', 'hp_dev_3245', 'hp_dev_4237']


## 3. 构造 relation-looking `bridge` candidate pool


In [5]:
def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text.strip().lower())


RELATION_PATTERNS = {
    term: re.compile(r'(?<![A-Za-z])' + re.escape(term) + r'(?![A-Za-z])', re.IGNORECASE)
    for term in RELATION_TERMS
}


def matched_relation_terms(question: str):
    return [term for term, pattern in RELATION_PATTERNS.items() if pattern.search(question)]


raw_candidates = []
seen_questions = set()

for idx, ex in enumerate(hotpot):
    task_id = task_id_from_index(idx)
    if task_id in current_bridge_members:
        continue

    raw_type = ex.get('type', '')
    if raw_type != 'bridge':
        continue

    question = ex['question']
    terms = matched_relation_terms(question)
    if not terms:
        continue

    q_norm = normalize_text(question)
    if q_norm in seen_questions:
        continue
    seen_questions.add(q_norm)

    support = ex.get('supporting_facts', {})
    support_titles = support.get('title', []) if isinstance(support, dict) else []
    context = ex.get('context', {})
    context_titles = context.get('title', []) if isinstance(context, dict) else []

    raw_candidates.append({
        'idx': idx,
        'task_id': task_id,
        'question': question,
        'answer': ex['answer'],
        'raw_type': raw_type,
        'level': ex.get('level', ''),
        'matched_relation_terms': '|'.join(terms),
        'support_titles': '|'.join(support_titles),
        'context_titles_preview': '|'.join(context_titles[:6]),
    })

raw_candidates = raw_candidates[:RAW_CANDIDATE_SIZE]
filtered_candidates = list(raw_candidates)

print('raw candidates =', len(raw_candidates))
print('filtered candidates =', len(filtered_candidates))
print('relation-term distribution =', Counter(term for row in raw_candidates for term in row['matched_relation_terms'].split('|') if term))


raw candidates = 15
filtered candidates = 15
relation-term distribution = Counter({'father': 5, 'daughter': 4, 'wife': 3, 'son': 2, 'mother': 1, 'husband': 1})


## 4. 写出第一批扩池文件


In [6]:
raw_csv_path = RESULTS_DIR / 'candidate_batch_raw.csv'
filtered_csv_path = RESULTS_DIR / 'candidate_batch_filtered.csv'
annotation_csv_path = RESULTS_DIR / 'candidate_batch_for_subtype_annotation.csv'
full_json_path = RESULTS_DIR / 'candidate_batch_full.json'
summary_md_path = RESULTS_DIR / 'candidate_batch_summary.md'


def write_csv(path, rows, fieldnames):
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


if raw_candidates:
    fieldnames = list(raw_candidates[0].keys())
    write_csv(raw_csv_path, raw_candidates, fieldnames)
    write_csv(filtered_csv_path, filtered_candidates, fieldnames)

annotation_rows = []
for row in filtered_candidates:
    annotation_rows.append({
        'task_id': row['task_id'],
        'question': row['question'],
        'answer': row['answer'],
        'raw_type': row['raw_type'],
        'level': row['level'],
        'matched_relation_terms': row['matched_relation_terms'],
        'support_titles': row['support_titles'],
        'reasoning_label': '',
        'bridge_subtype': '',
        'keep_drop': '',
        'note': '',
    })

if annotation_rows:
    write_csv(annotation_csv_path, annotation_rows, list(annotation_rows[0].keys()))

full_payload = []
for row in filtered_candidates:
    ex = hotpot[row['idx']]
    full_payload.append({
        **row,
        'full_example': {
            'question': ex['question'],
            'answer': ex['answer'],
            'type': ex.get('type', ''),
            'level': ex.get('level', ''),
            'supporting_facts': ex.get('supporting_facts', {}),
            'context': ex.get('context', {}),
        },
    })

with full_json_path.open('w') as f:
    json.dump(full_payload, f, ensure_ascii=False, indent=2)

summary_lines = [
    '# Relation-Chain Bridge Expansion Batch 1 Summary',
    '',
    f'- split: `{CURRENT_SPLIT}`',
    f'- raw candidates: `{len(raw_candidates)}`',
    f'- filtered candidates: `{len(filtered_candidates)}`',
    '',
    '## Relation-Term Distribution',
    '',
]
for term, count in Counter(term for row in raw_candidates for term in row['matched_relation_terms'].split('|') if term).most_common():
    summary_lines.append(f'- `{term}`: {count}')

summary_lines.extend([
    '',
    '## Next Step',
    '',
    '在 `candidate_batch_for_subtype_annotation.csv` 中补：',
    '',
    '- `reasoning_label`',
    '- `bridge_subtype`',
    '- `keep_drop`',
    '- `note`',
])
summary_md_path.write_text('\n'.join(summary_lines) + '\n')

print('Wrote:', raw_csv_path)
print('Wrote:', filtered_csv_path)
print('Wrote:', annotation_csv_path)
print('Wrote:', full_json_path)
print('Wrote:', summary_md_path)


Wrote: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/08_relation_chain_bridge_expansion/candidate_batch_raw.csv
Wrote: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/08_relation_chain_bridge_expansion/candidate_batch_filtered.csv
Wrote: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/08_relation_chain_bridge_expansion/candidate_batch_for_subtype_annotation.csv
Wrote: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/08_relation_chain_bridge_expansion/candidate_batch_full.json
Wrote: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/08_relation_chain_bridge_expansion/candidate_batch_summary.md


## 5. 预览 annotation 表


In [7]:
for row in annotation_rows[:5]:
    print(row['task_id'], '|', row['matched_relation_terms'], '|', row['question'])


hp_dev_0024 | father | What was the father of Kasper Schmeichel voted to be by the IFFHS in 1992?
hp_dev_0028 | father | Kaiser Ventures corporation was founded by an American industrialist who became known as the father of modern American shipbuilding?
hp_dev_0124 | daughter | The youngest daughter of Lady Mary-Gaye Curzon stars with Douglas Smith and Lucien Laviscount  in what 2017 film?
hp_dev_0198 | son | What amount was the settlement that the character from the Son of al Quada got in 2017?
hp_dev_0256 | wife | What year did Roy Rogers and his third wife star in a film directed by Frank McDonald?
